# Inverse Parameter Identification — Joint (D, κ) Inference

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cmhobbs96/pift-od-il-inverse-problems/blob/main/examples/08_inverse_parameters.ipynb)

This notebook reproduces **Example 3a, Figure 3** from Alberts & Bilionis (2023).

Consider the nonlinear PDE

$$D\phi''(x) - \kappa\phi^3(x) = f(x), \quad x \in [0, 1]$$

with $f(x) = \cos(4x)$ and zero Dirichlet BCs.  The parameters $(D, \kappa)$ are
**unknown** and must be inferred jointly with the field $\phi$ from $n_{\text{obs}} = 40$
noisy observations of $\phi$.

**Method:** nested SGLD (Algorithm 3) on the log-transformed outer variable
$\lambda = (\log D, \log\kappa)$ with Jeffrey's (flat log-space) priors.  An inner
SGLD chain samples $\phi$ from the conditional posterior given $\lambda$.

**Estimated runtime:** 30–60 minutes on CPU with the default CONFIG.

In [ ]:
%pip install -q git+https://github.com/cmhobbs96/pift-od-il-inverse-problems.git

import time
import jax
jax.config.update('jax_enable_x64', True)
import numpy as np
import matplotlib.pyplot as plt

from pipelines.phase_c_inverse_params import run_phase_c_inverse_params

print('JAX backend:', jax.default_backend(), '| devices:', jax.devices())

def show_fig(fig, dpi=120):
    import tempfile
    from IPython.display import Image, display
    with tempfile.NamedTemporaryFile(suffix='.png', delete=False) as f:
        fig.savefig(f.name, dpi=dpi, bbox_inches='tight')
        plt.close(fig)
        display(Image(f.name))

## Configuration

In [ ]:
CONFIG = {
    'D_true':               0.1,
    'kappa_true':           1.0,
    'beta':                 1e3,        # physics trust                          [1e2, 1e7]
    'n_obs':                40,
    'noise_std':            0.01,
    'K':                    20,
    'warmup_steps':         20000,
    'outer_steps':          5000,
    'outer_step_size0':     1e-5,       # outer alpha_0                          [1e-7, 1e-2]
    'inner_step_size0':     1e-7,       # inner alpha_0 approx 1/beta            [1e-8, 1e-2]
    'inner_T_prior':        10,
    'inner_T_posterior':    1,
    'n_quad':               1,
    'n_grid':               200,
    'burn_in_frac':         0.3,
    'max_condition_number': 100.0,
    'lambda0_log_D':        0.0,        # D_init = 1.0
    'lambda0_log_kappa':    0.5,        # kappa_init approx 1.65
}

## Run Nested SGLD

In [ ]:
t_start = time.perf_counter()

params_result = run_phase_c_inverse_params(
    cfg=CONFIG,
    device_preference=jax.default_backend(),
    save_outputs=False,
)

elapsed = time.perf_counter() - t_start
print(f'Status:  {params_result["status"]}')
print(f'Runtime: {elapsed:.1f} s  ({elapsed/60:.1f} min)')

D_post    = params_result.get('D_posterior', {})
kappa_post = params_result.get('kappa_posterior', {})
print(f'Posterior D:     mean={D_post.get("mean", float("nan")):.4f}  '
      f'std={D_post.get("std", float("nan")):.4f}  (true={CONFIG["D_true"]})')
print(f'Posterior kappa: mean={kappa_post.get("mean", float("nan")):.4f}  '
      f'std={kappa_post.get("std", float("nan")):.4f}  (true={CONFIG["kappa_true"]})')

## Results

In [ ]:
# Posterior summary for D and kappa
lambda_chain = np.asarray(params_result.get('lambda_chain', np.zeros((1, 2))))
burn_in = int(CONFIG['outer_steps'] * CONFIG['burn_in_frac'])
post_chain = lambda_chain[burn_in:]

if post_chain.shape[0] > 0 and post_chain.shape[1] >= 2:
    D_samples     = np.exp(post_chain[:, 0])
    kappa_samples = np.exp(post_chain[:, 1])

    D_med   = float(np.median(D_samples))
    D_q05   = float(np.percentile(D_samples, 5))
    D_q95   = float(np.percentile(D_samples, 95))
    k_med   = float(np.median(kappa_samples))
    k_q05   = float(np.percentile(kappa_samples, 5))
    k_q95   = float(np.percentile(kappa_samples, 95))

    print('Posterior quantile summary (post-burn-in):')
    print(f'  D:     median={D_med:.4f}   90% CI=[{D_q05:.4f}, {D_q95:.4f}]   true={CONFIG["D_true"]}')
    print(f'  kappa: median={k_med:.4f}   90% CI=[{k_q05:.4f}, {k_q95:.4f}]   true={CONFIG["kappa_true"]}')

    # Joint scatter and marginal histograms
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # D marginal
    axes[0].hist(D_samples, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[0].axvline(CONFIG['D_true'], color='tomato', lw=2, linestyle='--', label='Truth')
    axes[0].axvline(D_med, color='navy', lw=1.5, linestyle=':', label='Median')
    axes[0].set_xlabel('D')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Posterior D')
    axes[0].legend(fontsize=9)

    # kappa marginal
    axes[1].hist(kappa_samples, bins=40, color='darkorange', edgecolor='white', alpha=0.85)
    axes[1].axvline(CONFIG['kappa_true'], color='tomato', lw=2, linestyle='--', label='Truth')
    axes[1].axvline(k_med, color='saddlebrown', lw=1.5, linestyle=':', label='Median')
    axes[1].set_xlabel('\u03ba')
    axes[1].set_ylabel('Count')
    axes[1].set_title('Posterior \u03ba')
    axes[1].legend(fontsize=9)

    # Joint scatter
    axes[2].scatter(D_samples, kappa_samples, s=4, alpha=0.3, color='purple')
    axes[2].scatter([CONFIG['D_true']], [CONFIG['kappa_true']],
                    s=120, color='tomato', zorder=10, marker='*', label='Truth')
    axes[2].set_xlabel('D')
    axes[2].set_ylabel('\u03ba')
    axes[2].set_title('Joint Posterior (D, \u03ba)')
    axes[2].legend(fontsize=9)

    fig.suptitle('Inverse Parameter Identification — Posterior Distributions', fontsize=12)
    fig.tight_layout()
    show_fig(fig)

    # Lambda trace
    fig2, axes2 = plt.subplots(1, 2, figsize=(11, 3.5))
    axes2[0].plot(lambda_chain[:, 0], lw=0.7, color='steelblue', alpha=0.9)
    axes2[0].axhline(np.log(CONFIG['D_true']), color='tomato', lw=1.5,
                     linestyle='--', label='log(D_true)')
    axes2[0].set_xlabel('Outer step')
    axes2[0].set_ylabel('log(D)')
    axes2[0].set_title('\u03bb trace: log(D)')
    axes2[0].legend(fontsize=9)

    axes2[1].plot(lambda_chain[:, 1], lw=0.7, color='darkorange', alpha=0.9)
    axes2[1].axhline(np.log(CONFIG['kappa_true']), color='tomato', lw=1.5,
                     linestyle='--', label='log(kappa_true)')
    axes2[1].set_xlabel('Outer step')
    axes2[1].set_ylabel('log(\u03ba)')
    axes2[1].set_title('\u03bb trace: log(\u03ba)')
    axes2[1].legend(fontsize=9)

    fig2.suptitle('Outer Chain \u03bb Traces (check convergence toward truth)', fontsize=11)
    fig2.tight_layout()
    show_fig(fig2)
else:
    print('lambda_chain not available or too short for plotting.')

## Interpretation

**Expected results** with the default CONFIG ($D_\text{true}=0.1$, $\kappa_\text{true}=1.0$,
$\beta=10^3$):

- The posterior for $D$ should concentrate near $0.1$ and the posterior for $\kappa$ near $1.0$.
- The $\lambda$ trace plots should show the outer chain moving toward
  $\log D_\text{true} \approx -2.3$ and $\log\kappa_\text{true} = 0$ after warm-up.

With $\beta = 10^3$ and conservative step sizes the chain may need more outer steps to
fully converge.  If the traces are still drifting at the end of the run, try:
- Increasing `outer_steps` (e.g., 10 000).
- Slightly increasing `outer_step_size0` (e.g., `5e-5`).
- Increasing `inner_T_posterior` to get better inner mixing per outer step.